## Training models 

In [1]:
!pip install lightgbm 

In [2]:
#%pip install matplotlib
#%pip install numpy
#%pip install pandas
#%pip install xgboost


In [3]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [23]:
import json
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", message=".*eval_set.*")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
TARGET = "isFraud"
SPLIT_KEY = 'step'

#### Load features and feature set

In [5]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [6]:
df = pd.read_parquet(DATA_DIR/"paysim_features.parquet")
with open(DATA_DIR / "feature_sets.json", "r") as file:
    feature_set = json.load(file)

In [7]:

df.shape

(2770409, 10)

In [8]:
df.head()



,amount_to_oldbalance_ratio,dest_prior_txn_count,hour,is_night,is_transfer,log_amount,sender_near_empty,sender_zero_balance,isFraud,step
0,0.9945,0,1,1,1,5.2040,1,0,1,1
1,7.1688,0,1,1,1,11.8462,1,0,0,1
2,5.1445,0,1,1,0,11.7789,1,0,0,1
3,64.9368,0,1,1,1,13.4900,1,0,0,1
4,"315,325.2500",0,1,1,0,12.6614,0,1,0,1


#### Time based split -- train/test/val

In [9]:

df = df.sort_values(SPLIT_KEY, kind='stable').reset_index(drop=True)
train_end = df[SPLIT_KEY].quantile(0.70)
val_end = df[SPLIT_KEY].quantile(0.85)

train = df[df[SPLIT_KEY]<= train_end]
valid = df[(df[SPLIT_KEY]>train_end) & (df[SPLIT_KEY] <= val_end)]
test = df[df[SPLIT_KEY]>val_end]

In [10]:
summary = pd.DataFrame({
    "rows": [len(train), len(valid), len(test)],
    "frauds": [int(train[TARGET].sum()), int(valid[TARGET].sum()), int(test[TARGET].sum())],
    "fraud_rate_%": [train[TARGET].mean() * 100, valid[TARGET].mean() * 100, test[TARGET].mean() * 100],
    "first_step": [train[SPLIT_KEY].min(), valid[SPLIT_KEY].min(), test[SPLIT_KEY].min()],
    "last_step": [train[SPLIT_KEY].max(), valid[SPLIT_KEY].max(), test[SPLIT_KEY].max()],
}, index=["train", "valid", "test"])
summary

,rows,frauds,fraud_rate_%,first_step,last_step
train,1951895,3643,0.1866,1,323
valid,410437,560,0.1364,324,377
test,408077,4010,0.9827,378,743


In [11]:
df.head()

,amount_to_oldbalance_ratio,dest_prior_txn_count,hour,is_night,is_transfer,log_amount,sender_near_empty,sender_zero_balance,isFraud,step
0,0.9945,0,1,1,1,5.2040,1,0,1,1
1,7.1688,0,1,1,1,11.8462,1,0,0,1
2,5.1445,0,1,1,0,11.7789,1,0,0,1
3,64.9368,0,1,1,1,13.4900,1,0,0,1
4,"315,325.2500",0,1,1,0,12.6614,0,1,0,1


#### split verify

In [12]:
assert train[SPLIT_KEY].max() < valid[SPLIT_KEY].min(), "train and valid overlap in time"
assert valid[SPLIT_KEY].max() < test[SPLIT_KEY].min(), "valid and test overlap in time"
assert len(train) + len(valid) + len(test) == len(df), "rows lost in the split"

y_train, y_valid, y_test = train[TARGET], valid[TARGET], test[TARGET]

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
SCALE_POS_WEIGHT = neg / pos
print(f"train legit: {neg:,} | train fraud: {pos:,} | scale_pos_weight = {SCALE_POS_WEIGHT:,.1f}")

train legit: 1,948,252 | train fraud: 3,643 | scale_pos_weight = 534.8


#### one evaluation helper for every model

In [13]:
def evaluate(y_true, scores, threshold=0.5):
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    return {
        "pr_auc": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }


results = []


def record(name, feature_set, scores, threshold=0.5):
    results.append({"model": name, "features": feature_set, **evaluate(y_valid, scores, threshold)})

## Baseline model: Logistic Regression 

In [14]:
logreg_scores = {}
for set_name, cols in feature_set.items():
    logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced",max_iter=1000,random_state=RANDOM_STATE)
    )
    logreg.fit(train[cols],y_train)
    logreg_scores[set_name] = logreg.predict_proba(valid[cols])[:,1]
    record("logreg balance",set_name,logreg_scores[set_name])
pd.DataFrame(results).round(4)

,model,features,pr_auc,roc_auc,precision,recall,f1,tp,fp,fn,tn
0,logreg balance,STRICT,0.0902,0.8319,0.0059,0.7375,0.0117,413,69463,147,340414
1,logreg balance,STANDARD,0.1138,0.9050,0.0042,0.8661,0.0084,485,114943,75,294934


#### Xgboost model training 

In [15]:
def fit_xgb(feature_names, **overrides):
    params = dict(
        n_estimators=1000, learning_rate=0.1, max_depth=6, min_child_weight=1,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=SCALE_POS_WEIGHT,
        eval_metric="aucpr", early_stopping_rounds=50, tree_method="hist",
        n_jobs=-1, random_state=RANDOM_STATE,
    )
    params.update(overrides)
    model = xgb.XGBClassifier(**params)
    model.fit(train[feature_names], y_train,
              eval_set=[(valid[feature_names], y_valid)], verbose=False)
    return model


xgb_models, xgb_scores = {}, {}
for set_name, cols in feature_set.items():
    xgb_models[set_name] = fit_xgb(cols)
    xgb_scores[set_name] = xgb_models[set_name].predict_proba(valid[cols])[:, 1]
    record("xgboost", set_name, xgb_scores[set_name])
    print(f"{set_name:<9} best_iteration = {xgb_models[set_name].best_iteration}")

pd.DataFrame(results).round(4)

STRICT    best_iteration = 62
STANDARD  best_iteration = 93


,model,features,pr_auc,roc_auc,precision,recall,f1,tp,fp,fn,tn
0,logreg balance,STRICT,0.0902,0.8319,0.0059,0.7375,0.0117,413,69463,147,340414
1,logreg balance,STANDARD,0.1138,0.9050,0.0042,0.8661,0.0084,485,114943,75,294934
2,xgboost,STRICT,0.3760,0.9294,0.0227,0.8054,0.0441,451,19433,109,390444
3,xgboost,STANDARD,0.9390,0.9999,0.4316,0.9964,0.6023,558,735,2,409142


#### TRAIN LIGHT GBM

In [16]:
def fit_lgbm(feature_names, **overrides):
    params = dict(
        n_estimators=1000, learning_rate=0.1, num_leaves=63, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, reg_lambda=5.0, scale_pos_weight=SCALE_POS_WEIGHT,
        metric="None", n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
    )
    params.update(overrides)
    model = lgb.LGBMClassifier(**params)
    model.fit(train[feature_names], y_train,
              eval_set=[(valid[feature_names], y_valid)], eval_metric="average_precision",
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    return model


lgbm_models, lgbm_scores = {}, {}
for set_name, cols in feature_set.items():
    lgbm_models[set_name] = fit_lgbm(cols)
    lgbm_scores[set_name] = lgbm_models[set_name].predict_proba(valid[cols])[:, 1]
    record("lightgbm", set_name, lgbm_scores[set_name])

pd.DataFrame(results).round(4)

,model,features,pr_auc,roc_auc,precision,recall,f1,tp,fp,fn,tn
0,logreg balance,STRICT,0.0902,0.8319,0.0059,0.7375,0.0117,413,69463,147,340414
1,logreg balance,STANDARD,0.1138,0.9050,0.0042,0.8661,0.0084,485,114943,75,294934
2,xgboost,STRICT,0.3760,0.9294,0.0227,0.8054,0.0441,451,19433,109,390444
3,xgboost,STANDARD,0.9390,0.9999,0.4316,0.9964,0.6023,558,735,2,409142
4,lightgbm,STRICT,0.0075,0.7339,0.0024,0.7375,0.0048,413,171197,147,238680
5,lightgbm,STANDARD,0.4541,0.9834,0.3846,0.9464,0.5470,530,848,30,409029


In [17]:
comparison = (pd.DataFrame(results)
              .pivot_table(index="model", columns="features", values="pr_auc")
              .assign(gain=lambda t: t["STANDARD"] - t["STRICT"]))
comparison.round(4)

features,STANDARD,STRICT,gain
model,,,
lightgbm,0.4541,0.0075,0.4466
logreg balance,0.1138,0.0902,0.0236
xgboost,0.9390,0.3760,0.5630


In [25]:
FINAL_FEATURE_SET = "STANDARD"
FINAL_MODEL_NAME = "xgboost"
final_model = xgb_models[FINAL_FEATURE_SET]
final_cols = feature_set[FINAL_FEATURE_SET]


#### Choose the decision on the validation set

In [19]:
valid_scores = final_model.predict_proba(valid[final_cols])[:, 1]
precision, recall, thresholds = precision_recall_curve(y_valid, valid_scores)
f1_curve = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)

best_f1_idx = int(np.argmax(f1_curve))
THRESHOLD_F1 = float(thresholds[best_f1_idx])

TARGET_PRECISION = 0.90
meets_precision = precision[:-1] >= TARGET_PRECISION
if meets_precision.any():
    candidate_idx = np.where(meets_precision)[0]
    THRESHOLD_PRECISION = float(thresholds[candidate_idx[np.argmax(recall[:-1][candidate_idx])]])
else:
    THRESHOLD_PRECISION = THRESHOLD_F1

threshold_table = pd.DataFrame(
    {name: evaluate(y_valid, valid_scores, t)
     for name, t in {"default 0.5": 0.5,
                     "max F1": THRESHOLD_F1,
                     f"precision >= {TARGET_PRECISION:.0%}": THRESHOLD_PRECISION}.items()}
).T
threshold_table["threshold"] = [0.5, THRESHOLD_F1, THRESHOLD_PRECISION]
threshold_table.round(4)

,pr_auc,roc_auc,precision,recall,f1,tp,fp,fn,tn,threshold
default 0.5,0.9390,0.9999,0.4316,0.9964,0.6023,558.0000,735.0000,2.0000,"409,142.0000",0.5000
max F1,0.9390,0.9999,0.9374,0.8286,0.8796,464.0000,31.0000,96.0000,"409,846.0000",0.9968
precision >= 90%,0.9390,0.9999,0.9013,0.8482,0.8740,475.0000,52.0000,85.0000,"409,825.0000",0.9952


#### here we are considering threshold as maxf1 - 0.9968 (THRESHOLD F1)

In [20]:
FINAL_THRESHOLD = THRESHOLD_F1

#### FINAL TEST EVALUATION

In [ ]:
test_scores = final_model.predict_proba(test[final_cols])[:, 1]
test_metrics = evaluate(y_test, test_scores, FINAL_THRESHOLD)
valid_metrics = evaluate(y_valid, valid_scores, FINAL_THRESHOLD)

pd.DataFrame({"validation": valid_metrics, "test": test_metrics}).round(4)

,validation,test
pr_auc,0.9390,0.9871
roc_auc,0.9999,0.9999
precision,0.9374,0.9867
recall,0.8286,0.8504
f1,0.8796,0.9135
tp,464.0000,"3,410.0000"
fp,31.0000,46.0000
fn,96.0000,600.0000
tn,"409,846.0000","404,021.0000"


In [22]:
cm = confusion_matrix(y_test, (test_scores >= FINAL_THRESHOLD).astype(int), labels=[0, 1])
cm_df = pd.DataFrame(cm, index=["actual legit", "actual fraud"],
                     columns=["predicted legit", "predicted fraud"])
print(cm_df)
print()
print(f"Of {int(y_test.sum())} real frauds we caught {test_metrics['tp']} and missed {test_metrics['fn']}.")
print(f"We raised {test_metrics['fp']} false alarms for every {test_metrics['tp']} correct flags "
      f"(precision {test_metrics['precision']:.1%}).")

              predicted legit  predicted fraud
actual legit           404021               46
actual fraud              600             3410

Of 4010 real frauds we caught 3410 and missed 600.
We raised 46 false alarms for every 3410 correct flags (precision 98.7%).


#### Save the model and everything needed to be used it later

In [26]:
final_model.save_model(MODEL_DIR / "xgb_fraud.json")

meta = {
    "model": FINAL_MODEL_NAME,
    "feature_set": FINAL_FEATURE_SET,
    "features": final_cols,
    "threshold": FINAL_THRESHOLD,
    "scale_pos_weight": float(SCALE_POS_WEIGHT),
    "best_iteration": int(final_model.best_iteration),
    "split": {"train_end_step": float(train_end), "valid_end_step": float(val_end)},
    "validation_metrics": {k: float(v) for k, v in valid_metrics.items()},
    "test_metrics": {k: float(v) for k, v in test_metrics.items()},
}
(MODEL_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))
pd.DataFrame(results).to_csv(MODEL_DIR / "validation_results.csv", index=False)

reloaded = xgb.XGBClassifier()
reloaded.load_model(MODEL_DIR / "xgb_fraud.json")
check = reloaded.predict_proba(test[final_cols].head(1000))[:, 1]
assert np.allclose(check, test_scores[:1000], atol=1e-6), "reloaded model gives different scores"
print("saved:", sorted(p.name for p in MODEL_DIR.iterdir()))

saved: ['model_meta.json', 'validation_results.csv', 'xgb_fraud.json']
